In [18]:
import pandas as pd
import numpy as np
import joblib

## 4. topic vs variant analysis 

THis notebook will discover whether a trained model has actually learned from pure grammatical features of both variant of Portuguese. In order to do so, a corpus was created by manually collecting snipet of news articles.
Distributions are as follows: 
- 18 Portuguese news articles covering international topic 
- 12 Portuguese news articles covering domestic topic 
- 15 Brazilian news articles covering international topic 
- 15 Brazilian news articles covering domestic topic 

In [ ]:
corpus_df = pd.read_csv('path_corpus_v1.csv')
corpus_df[[
        "article_id",
        "publication",
        "publication_date",
        "variety_gold",
        "geographical_scope",
        "word_count"
    ]].head()

,article_id,publication,publication_date,variety_gold,geographical_scope,word_count
0,pt_1,jornal de noticias,2026-11-10,pt-pt,domestic,102
1,pt_2,rtp,2026-11-10,pt-pt,domestic,188
2,pt_3,rtp,2026-11-10,pt-pt,domestic,109
3,pt_4,rtp,2026-11-10,pt-pt,domestic,104
4,pt_5,rtp,2026-11-06,pt-pt,domestic,151


In [34]:
# Missing values
print("Missing values:")
display(corpus_df.isnull().sum())

# Variety distribution
print("\nVariety distribution:")
display(corpus_df["variety_gold"].value_counts())

# Geographical scope distribution
print("\nGeographical scope distribution:")
display(corpus_df["geographical_scope"].value_counts())

# Variety × geographical scope
print("\nVariety and geographical scope:")
display(pd.crosstab(
    corpus_df["variety_gold"],
    corpus_df["geographical_scope"]
))

Missing values:


article_id             0
publication            0
publication_country    0
url                    0
publication_date       0
geographical_scope     0
variety_gold           0
text                   0
word_count             0
dtype: int64


Variety distribution:


variety_gold
pt-pt    30
pt-br    30
Name: count, dtype: int64


Geographical scope distribution:


geographical_scope
international    33
domestic         27
Name: count, dtype: int64


Variety and geographical scope:


geographical_scope,domestic,international
variety_gold,,
pt-br,15,15
pt-pt,12,18


In [35]:
import re
import unicodedata
import pandas as pd 


In [36]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize Unicode
    text = unicodedata.normalize("NFKD", text)

    # Remove diacritics
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    # Lowercase
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [37]:
corpus_df['text_clean'] = corpus_df['text'].apply(clean_text)

In [38]:
combined_model = joblib.load('../models/combined_tfidf_logistic_regression.joblib')

In [39]:
print(combined_model)

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word',
                                                 TfidfVectorizer(max_features=100000,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=200000,
                                                                 min_df=3,
                                                                 ngram_range=(3,
                                                                              5),
                                         

In [40]:
print(combined_model.classes_)

[0 1]


In [41]:
X_external = corpus_df['text_clean']

label_mapping = { 
    "pt-pt": 0, 
    "pt-br": 1, 
}

y_external = corpus_df['variety_gold'].map(label_mapping)

print("label distribution")

print(y_external.value_counts())

label distribution
variety_gold
0    30
1    30
Name: count, dtype: int64


In [42]:
y_pred = combined_model.predict(X_external)

y_prob = combined_model.predict_proba(X_external)

corpus_df['prediction'] = y_pred 

inverse_mapping = { 
    0:"pt-pt", 
    1:"pt-br"
}

corpus_df['variety_predicted'] = corpus_df['prediction'].map(inverse_mapping)

display(corpus_df[[
    "article_id",
    "variety_gold",
    "variety_predicted",
    "geographical_scope"
]].head(30))

,article_id,variety_gold,variety_predicted,geographical_scope
0,pt_1,pt-pt,pt-pt,domestic
1,pt_2,pt-pt,pt-pt,domestic
2,pt_3,pt-pt,pt-pt,domestic
3,pt_4,pt-pt,pt-pt,domestic
4,pt_5,pt-pt,pt-pt,domestic
5,pt_6,pt-pt,pt-pt,domestic
6,pt_7,pt-pt,pt-pt,domestic
7,pt_8,pt-pt,pt-pt,domestic
8,pt_9,pt-pt,pt-pt,domestic
9,pt_10,pt-pt,pt-pt,domestic


In [43]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_external, y_pred)
macro_f1 = f1_score(y_external, y_pred, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_external,
    y_pred,
    target_names=["PT-PT", "PT-BR"]
))

Accuracy: 0.7500
Macro F1: 0.7483

Classification Report:
              precision    recall  f1-score   support

       PT-PT       0.80      0.67      0.73        30
       PT-BR       0.71      0.83      0.77        30

    accuracy                           0.75        60
   macro avg       0.76      0.75      0.75        60
weighted avg       0.76      0.75      0.75        60



In [44]:
from sklearn.metrics import accuracy_score, f1_score

# Evaluate performance by geographical scope
scope_results = []

for scope in ["domestic", "international"]:

    subset = corpus_df[
        corpus_df["geographical_scope"] == scope
    ]

    accuracy = accuracy_score(
        subset["variety_gold"].map(label_mapping),
        subset["prediction"]
    )

    macro_f1 = f1_score(
        subset["variety_gold"].map(label_mapping),
        subset["prediction"],
        average="macro"
    )

    scope_results.append({
        "geographical_scope": scope,
        "n_articles": len(subset),
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

scope_results_df = pd.DataFrame(scope_results)

display(scope_results_df)

,geographical_scope,n_articles,accuracy,macro_f1
0,domestic,27,1.000000,1.000000
1,international,33,0.545455,0.543779


In [45]:
group_results = []

for (variety, scope), group in corpus_df.groupby(
    ["variety_gold", "geographical_scope"]
):

    y_true = group["variety_gold"].map(label_mapping)
    y_pred_group = group["prediction"]

    group_results.append({
        "variety": variety,
        "geographical_scope": scope,
        "n_articles": len(group),
        "correct": (y_true == y_pred_group).sum(),
        "accuracy": accuracy_score(y_true, y_pred_group)
    })

group_results_df = pd.DataFrame(group_results)

display(group_results_df)

,variety,geographical_scope,n_articles,correct,accuracy
0,pt-br,domestic,15,15,1.000000
1,pt-br,international,15,10,0.666667
2,pt-pt,domestic,12,12,1.000000
3,pt-pt,international,18,8,0.444444
